# Exploration

```{important} AI Disclosure
The written content in this report was drafted by Jenny Lee and refined using Generative AI for clarity and grammar. All ideas, analysis decisions, and interpretations are original. The prompt used throughout was: *"Refine my words."*
```

```{note} Viewing the Context
The written content in this report was drafted by Jenny Lee and refined using Generative AI for clarity and grammar. All ideas, analysis decisions, and interpretations are original. The prompt used throughout was: *"Refine my words."*
```

## Data Collection

🔗 [Data Source](https://imdbapi.dev)

In [2]:
import pandas as pd
import numpy as np
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import json

## E1
> If you dataset has an unusual structure, describe the steps and
related challenges in extracting the dataset. For example, if your data comes from an API, talk about the
API calls you had to run in order to get the data.

In [17]:
BASE_URL = "https://api.imdbapi.dev/"
response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear>=2024")
data = response.json()
df = pd.DataFrame(data["titles"])
pagetoken = data["nextPageToken"]
time.sleep(1)  

for page in range(1, 20):
      try:
            response = requests.get(f"{BASE_URL}/titles?types=MOVIE&startYear=2026&pageToken={pagetoken}", timeout=10)
            
            if response.status_code == 429:
                  wait_time = int(response.headers.get("Retry-After", 5))
                  print(f"Page {page}: Rate limited! Waiting {wait_time}s...")
                  time.sleep(wait_time)
                  continue
            
            response.raise_for_status()
            
            if not response.text:
                  break
            
            data = response.json()
            
            if "titles" not in data or not data["titles"]:
                  break
                  
            df_page = pd.DataFrame(data["titles"])
            df = pd.concat([df, df_page], ignore_index=True)
            
            if "nextPageToken" in data and data["nextPageToken"]:
                  pagetoken = data["nextPageToken"]
            else:
                  break
            
            time.sleep(1)  
      except Exception as e:
            break

print(f"Final shape: {df.shape}")
print(f"Total Number of rows: {len(df)}")
display(df.head(10))

Final shape: (1000, 10)
Total Number of rows: 1000


,id,type,primaryTitle,originalTitle,primaryImage,startYear,runtimeSeconds,genres,rating,plot
0,tt12042730,movie,Project Hail Mary,Project Hail Mary,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,9360.0,"[Adventure, Comedy, Sci-Fi]","{'aggregateRating': 8.4, 'voteCount': 178625}",A science teacher wakes up alone on a spaceshi...
1,tt28650488,movie,The Super Mario Galaxy Movie,The Super Mario Galaxy Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5880.0,"[Animation, Adventure, Comedy, Family, Fantasy]","{'aggregateRating': 6.5, 'voteCount': 25522}","Mario ventures into space, exploring cosmic wo..."
2,tt32430579,movie,Crime 101,Crime 101,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,8400.0,"[Crime, Drama, Thriller]","{'aggregateRating': 6.8, 'voteCount': 48321}","An elusive thief, eyeing his final score, enco..."
3,tt33071426,movie,The Drama,The Drama,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6300.0,"[Comedy, Drama, Romance]","{'aggregateRating': 7.5, 'voteCount': 17072}",A happily engaged couple is put to the test wh...
4,tt37209937,movie,Pizza Movie,Pizza Movie,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,5820.0,[Comedy],"{'aggregateRating': 5.8, 'voteCount': 4258}",High college students face an unexpectedly epi...
5,tt27543632,movie,The Housemaid,The Housemaid,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,7860.0,"[Drama, Mystery, Thriller]","{'aggregateRating': 6.8, 'voteCount': 128853}",A struggling young woman is relieved by the ch...
6,tt33244668,movie,Anaconda,Anaconda,{'url': 'https://m.media-amazon.com/images/M/M...,2025.0,5940.0,"[Action, Adventure, Comedy]","{'aggregateRating': 5.6, 'voteCount': 52429}",A group of friends are going through a mid-lif...
7,tt27552099,movie,Mike & Nick & Nick & Alice,Mike & Nick & Nick & Alice,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,6420.0,"[Action, Comedy, Crime]","{'aggregateRating': 6.2, 'voteCount': 12231}",Two friends navigate the dangerous world of or...
8,tt39139925,movie,Dhurandhar The Revenge,Dhurandhar The Revenge,{'url': 'https://m.media-amazon.com/images/M/M...,2026.0,13800.0,"[Action, Crime, Thriller]","{'aggregateRating': 8.5, 'voteCount': 43693}",Jaskirat Singh Rangi descends deeper into his ...
9,tt0049833,movie,The Ten Commandments,The Ten Commandments,{'url': 'https://m.media-amazon.com/images/M/M...,1956.0,13200.0,"[Adventure, Drama, Family, History]","{'aggregateRating': 7.9, 'voteCount': 84370}","Moses, raised as a prince of Egypt in the Phar..."


Data was collected across multiple endpoints for various purposes outlined throughout this report, with the `/titles` endpoint serving as the main source for general movie metadata. Results were paginated at 50 entries per page, with each response returning a token to access the next page; this pagination mechanism was used to retrieve a total of 1,000 rows filtered by `genre: MOVIE`.

The primary challenge in data extraction was managing the API's rate limits. To mitigate this, deliberate delays were introduced in the retrieval code, by pausing for 1 second between requests and extending to 5 seconds upon failure, ensuring stable and uninterrupted data collection.

```{code-cell} python
:linenos:

def fetch_box_office(ind, row, base_url, delay=0.5):
    title_id = row["id"]
    time.sleep(delay)  
    
    try:
        response = requests.get(f"{base_url}/titles/{title_id}/boxOffice", timeout=5)
        
        if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited on {title_id}, waiting {wait_time}s...")
            time.sleep(wait_time)
            return fetch_box_office(ind, row, base_url, delay=2)
        
        response.raise_for_status() 
        data = response.json()
        worldwide_gross = data["worldwideGross"]["amount"] if "worldwideGross" in data else None
        worldwide_gross_currency = data["worldwideGross"]["currency"] if "worldwideGross" in data else None
        production_budget = data["productionBudget"]["amount"] if "productionBudget" in data else None
        production_budget_currency = data["productionBudget"]["currency"] if "productionBudget" in data else None

        return ind, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, None, data 

skipped_count = 0
printed_sample = False

for ind, row in df.iterrows():
    ind_result, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, error, data = fetch_box_office(ind, row, BASE_URL)

    if not printed_sample and not error and data:
        print("Sample API response to show nested data:")
        print(data)
        printed_sample = True

    if error:
        skipped_count += 1
    else:
        df.at[ind, "worldwideGross"] = worldwide_gross
        df.at[ind, "worldwideGrossCurrency"] = worldwide_gross_currency
        df.at[ind, "productionBudget"] = production_budget
        df.at[ind, "productionBudgetCurrency"] = production_budget_currency

print(f"Skipped {skipped_count} titles due to errors.")
print(f"Box office data added to {len(df) - skipped_count} rows.")

display(df.head(10))

```

In [ ]:
def fetch_box_office(ind, row, base_url, delay=0.5):
      title_id = row["id"]
      time.sleep(delay)  

      response = requests.get(f"{base_url}/titles/{title_id}/boxOffice", timeout=5)
      
      if response.status_code == 429:
            wait_time = int(response.headers.get("Retry-After", 10))
            print(f"Rate limited on {title_id}, waiting {wait_time}s...")
            time.sleep(wait_time)

      return fetch_box_office(ind, row, base_url, delay=2)
        
      response.raise_for_status() 
      data = response.json()
      worldwide_gross = data["worldwideGross"]["amount"] if "worldwideGross" in data else None
      worldwide_gross_currency = data["worldwideGross"]["currency"] if "worldwideGross" in data else None
      production_budget = data["productionBudget"]["amount"] if "productionBudget" in data else None
      production_budget_currency = data["productionBudget"]["currency"] if "productionBudget" in data else None

        return ind, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, None, data 

skipped_count = 0
printed_sample = False

for ind, row in df.iterrows():
    ind_result, worldwide_gross, worldwide_gross_currency, production_budget, production_budget_currency, error, data = fetch_box_office(ind, row, BASE_URL)

    if not printed_sample and not error and data:
        print("Sample API response to show nested data:")
        print(data)
        printed_sample = True

    if error:
        skipped_count += 1
    else:
        df.at[ind, "worldwideGross"] = worldwide_gross
        df.at[ind, "worldwideGrossCurrency"] = worldwide_gross_currency
        df.at[ind, "productionBudget"] = production_budget
        df.at[ind, "productionBudgetCurrency"] = production_budget_currency

print(f"Skipped {skipped_count} titles due to errors.")
print(f"Box office data added to {len(df) - skipped_count} rows.")

display(df.head(10))

SyntaxError: expected 'except' or 'finally' block (2188525202.py, line 23)

The primary goal of this project is to identify key movie characteristics that drive a high worldwide gross relative to production budget, and to leverage those characteristics to build a predictive model for the gross-to-budget ratio. To retrieve the necessary financial data, the `/titles/{titleId}/boxOffice` endpoint was used, where `titleId` corresponds to the `id` column in the dataset.

## E2
> If you dataset has an unusual structure, describe the steps and related challenges in transforming and loading the dataset. For example, if your dataset is too large to
load all at once, talk about batch loading and/or the bigmemory package. If your dataset is images, talk
about file formatting issues and how you look at individual pixels. If your dataset is nested data frames,
talk about how you flatten and simplify that that.

As shown in the code above (lines 16–19), the data retrieved from `/titles/{titleId}/boxOffice` is structured as a nested dictionary. In particular, both `worldwideGross` and `productionBudget` contain subfields, requiring separate extraction of the amount and currency values.

In [13]:
missing_counts = df.isna().sum()
missing_counts[missing_counts > 0]

primaryImage          4
startYear             5
runtimeSeconds       36
rating               52
plot                  2
worldwideGross      138
productionBudget    232
dtype: int64

In [12]:
missing_df = df[df["worldwideGross"].isna() | df["productionBudget"].isna()]
missing_df["startYear"].value_counts()

startYear
2026.0    98
2025.0    83
2024.0    12
2027.0    10
2023.0     9
2020.0     8
2022.0     7
1973.0     2
2021.0     2
2013.0     2
2017.0     2
2011.0     2
2029.0     1
2018.0     1
1979.0     1
2019.0     1
2000.0     1
1975.0     1
1968.0     1
2007.0     1
2010.0     1
1965.0     1
1985.0     1
1961.0     1
2004.0     1
2002.0     1
1962.0     1
Name: count, dtype: int64